### 01. Setup

In [1]:
import os
from pathlib import Path
import random


import numpy as np
import matplotlib.pyplot as plt


import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

#### Reproducibility seeds

In [2]:
def set_seed(seed: int = 42) -> None:
    """Seed Python, NumPy, and PyTorch (CPU & all GPUs) and
    make cuDNN run in deterministic mode."""
    # ---- Python and NumPy -------------------------------------------
    random.seed(seed)
    np.random.seed(seed)

    # ---- PyTorch (CPU  &  GPU) --------------------------------------
    torch.manual_seed(seed)            
    torch.cuda.manual_seed_all(seed)   

    # ---- cuDNN: force repeatable convolutions -----------------------
    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark     = False

In [3]:
set_seed(15)

PyTorch offsets each worker’s seed by default, injecting new randomness. For reproducible results, you want workers to start from **fixed** seeds so every data-augmentation decision (flip, crop, colour-jitter) is repeatable across runs. The `worker_init_fn` function re-seeds Python, NumPy, and PyTorch CPU random-number generators inside **each** worker using a simple deterministic formula (`SEED + worker_id`).  The result will be identical batches, identical gradients, and identical model checkpoints, run after run.

In [4]:
def worker_init_fn_(worker_id: int) -> None:
    """Re-seed each DataLoader worker so their RNGs don't collide."""
    worker_seed = 42 + worker_id
    np.random.seed(worker_seed) 
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)

#### Data location

In [5]:
# make sure the data is downloaded,  extracted and split into train and test - if not please use the 01 notebook.
train_dir = Path("./data/train")
val_dir = Path("./data/test")

#### Hyperparameters and device

In [6]:
img_size = 64
batch_size = 32
lr = 0.001
epochs = 10 # set to low number for convenience. Change as needed.
model_name = "ai_capstone_pytorch_state_dict.pth"
num_classes = 2 # number of classes in the dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used is {device}")

Device used is cuda


### 01. Create the datasets

For datasets that follow a standard root/class_name/xxx.ext directory structure, PyTorch provides torchvision.datasets.ImageFolder. This is a specialized Dataset class that does all the work of finding image paths and inferring class labels for you. It's the most convenient way to load image datasets organized in folders, where each subfolder represents a different class. 

Custom transformations can be given as a parameter to the dataset creation to augment the collected data.


In cases when special treatment is needed one can create a custom dataset class inheriting from DataSet and customize the constructor, __getitem__ and __len__ methods: 

```python
class CustomBinaryClassDataset(Dataset):
    """A custom dataset for our agricultural land classification task."""

    def __init__(self, non_agri_dir, agri_dir, transform=None):
        """
        Args:
            non_agri_dir (Path): Directory with all the non-agricultural images.
            agri_dir (Path): Directory with all the agricultural images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Load non-agri paths and assign label 0
        for image_path in non_agri_dir.iterdir():
            self.image_paths.append(image_path)
            self.labels.append(0)

        # Load agri paths and assign label 1
        for image_path in agri_dir.iterdir():
            self.image_paths.append(image_path)
            self.labels.append(1)

        temp = list(zip(self.image_paths, self.labels))
        np.random.shuffle(temp)
        self.image_paths, self.labels = zip(*temp)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")

        label = self.labels[idx]

        # Apply transformations if they exist
        if self.transform:
            image = self.transform(image)

        return image, label
```


In [7]:
train_transform = transforms.Compose([ #transforms.Resize((64, 64)), # not really needed here as pictures are already 64x64, but potentially useful syntax
                                     transforms.RandomHorizontalFlip(p=0.5), # probability 0.5
                                     transforms.RandomVerticalFlip(p=0.5),   # probability 0.5
                                     transforms.RandomRotation(45),
                                     transforms.RandomAffine(0, shear=0.2),
                                     transforms.ToTensor(), # Converts PIL Image to a tensor, reorders to (channels, height, width) and scales values to [0, 1]
                                     transforms.Normalize(mean = [0.5, 0.5, 0.5], std = [0.5, 0.5, 0.5]) # Normalize from [0, 1] to [-1, 1]
])

val_transform = transforms.Compose([ #transforms.Resize((64, 64)), # not really needed here as pictures are already 64x64, but potentially useful syntax
                                     transforms.ToTensor(), # Converts PIL Image to a tensor, reorders to (channels, height, width) and scales values to [0, 1]
                                     transforms.Normalize(mean = [0.5, 0.5, 0.5], std = [0.5, 0.5, 0.5]) # Normalize from [0, 1] to [-1, 1]
])

In [8]:
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transform)

In [9]:
print(f"Classes found by ImageFolder: {train_dataset.classes}")
print(f"Class to index mapping: {train_dataset.class_to_idx}")

Classes found by ImageFolder: ['class_0_non_agri', 'class_1_agri']
Class to index mapping: {'class_0_non_agri': 0, 'class_1_agri': 1}


### 02. Setting up the train and test data loaders

The dataloader can specify things like batch size and wether or not to shuffle (True during training, False during validation). Note the initialization function setting up the randomness feed.

In [10]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, worker_init_fn=worker_init_fn_)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, worker_init_fn=worker_init_fn_)

In [11]:
images, labels = next(iter(train_loader))

print(f"Images batch shape: {images.shape}") 
print(f"Labels batch shape: {labels.shape}")


Images batch shape: torch.Size([32, 3, 64, 64])
Labels batch shape: torch.Size([32])


The relevant code is under src to iport in future notebooks as follows:
```python
from src.dataset import create_dataloaders
train_loader, val_loader, test_loader = create_dataloaders(batch_size)
```